# 消融实验：主干与序列编码器

四组共享同一份 train/val 切分和全部训练超参，只改 `backbone_type` / `encoder_type`：

| 变体 | backbone | encoder |
|---|---|---|
| 纯 CNN | cnn | none |
| 纯 Transformer | direct | transformer |
| 纯 LSTM | direct | lstm |
| CNN + LSTM | cnn | lstm |

`direct` 主干只有一个 1×1 通道投影，不做长度压缩，token 数保持 995；只有 `cnn` 会经过两次 AvgPool（995 → 497 → 248）。因此纯 Transformer 的注意力矩阵是 CNN 路径的 16 倍，显存不足时请调小 batch_size。


## 1. 环境与数据包


In [ ]:
"""环境引导：定位项目根目录，按需 pip 安装缺失依赖。"""
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT_OVERRIDE = None
INSTALL_MISSING_DEPS = True
PIP_INDEX_URL = os.environ.get("PIP_INDEX_URL") or "https://pypi.tuna.tsinghua.edu.cn/simple"
PACKAGE_ARCHIVE_NAME = "ramanv2.zip"
DATA_ARCHIVE_NAME = "data.npz"
REQUIRED_MODULES = {
    "torch": "torch", "numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "tqdm": "tqdm", "pandas": "pandas", "seaborn": "seaborn",
    "yaml": "PyYAML", "umap": "umap-learn", "openpyxl": "openpyxl",
}

def _has_payload(directory: Path) -> bool:
    return (directory / PACKAGE_ARCHIVE_NAME).is_file() or (directory / "ramanv2" / "core" / "config.py").is_file()

override = PROJECT_ROOT_OVERRIDE or os.environ.get("RAMAN_PROJECT_ROOT")
if override:
    PROJECT_ROOT = Path(override).expanduser().resolve()
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = next((c for c in (cwd, *cwd.parents) if _has_payload(c)), cwd)

missing = [pkg for mod, pkg in REQUIRED_MODULES.items() if importlib.util.find_spec(mod) is None]
if missing and INSTALL_MISSING_DEPS:
    command = [sys.executable, "-m", "pip", "install", "--no-cache-dir"]
    if PIP_INDEX_URL:
        command += ["--index-url", PIP_INDEX_URL]
    subprocess.run(command + missing, shell=False, check=True)
print(f"project_root = {PROJECT_ROOT}")


In [ ]:
# 输入检查：代码包必需；data.npz 可选，dataset/<数据集>/ 已在位时可跳过恢复。
REQUIRED_ARCHIVES = {"ramanv2.zip", DATA_ARCHIVE_NAME}
absent_archives = sorted(
    name for name in REQUIRED_ARCHIVES if not (PROJECT_ROOT / name).is_file()
)
if PACKAGE_ARCHIVE_NAME in absent_archives and not (PROJECT_ROOT / "ramanv2").is_dir():
    raise FileNotFoundError(
        f"缺少代码包：请把 {PACKAGE_ARCHIVE_NAME} 或解压后的 ramanv2/ 放到 {PROJECT_ROOT}"
    )
if DATA_ARCHIVE_NAME in absent_archives:
    print(f"未找到 {DATA_ARCHIVE_NAME}：若 dataset/<数据集>/init 已在位可忽略。")
print("输入检查完成。")


In [ ]:
from pathlib import Path
import shutil
import sys
import tempfile
import zipfile

PACKAGE_ARCHIVE = PROJECT_ROOT / PACKAGE_ARCHIVE_NAME
PACKAGE_DIR = PROJECT_ROOT / "ramanv2"


def load_package():
    """检测到压缩包时替换 ramanv2；否则使用已解压的包。"""
    if PACKAGE_ARCHIVE.is_file():
        temp_dir = Path(tempfile.mkdtemp(prefix="ramanv2_unpack_", dir=PROJECT_ROOT))
        try:
            with zipfile.ZipFile(PACKAGE_ARCHIVE) as archive:
                archive.extractall(temp_dir)
            unpacked_dir = temp_dir / PACKAGE_DIR.name
            if not unpacked_dir.is_dir():
                raise FileNotFoundError("压缩文件中未找到 ramanv2/ 目录")
            if PACKAGE_DIR.exists():
                if not PACKAGE_DIR.is_dir():
                    raise FileExistsError(f"包路径不是目录：{PACKAGE_DIR}")
                shutil.rmtree(PACKAGE_DIR)
            shutil.move(str(unpacked_dir), str(PROJECT_ROOT))
            print(f"replaced package: {PACKAGE_ARCHIVE.name} -> {PACKAGE_DIR}")
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)
    elif not PACKAGE_DIR.is_dir():
        raise FileNotFoundError(f"请先将 {PACKAGE_ARCHIVE_NAME} 放到 {PROJECT_ROOT}")
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))


load_package()


In [ ]:
import importlib
import sys


def reload_ramanv2():
    """清除 ramanv2 模块缓存，以加载当前目录中的最新代码。"""
    for name in list(sys.modules):
        if name == "ramanv2" or name.startswith("ramanv2."):
            del sys.modules[name]
    importlib.invalidate_caches()
    config_module = importlib.import_module("ramanv2.core.config")
    print("ramanv2 modules reloaded.")
    return config_module


config_module = reload_ramanv2()
build_config = config_module.build_config


In [ ]:
from ramanv2.packaging.platform_bundle import DATA_ARCHIVE_NAME, restore_data_archive

# 首次运行或更换 data.npz 后执行一次；会整体替换数据源的 init 与 init_test。
DATA_ARCHIVE = PROJECT_ROOT / DATA_ARCHIVE_NAME
if DATA_ARCHIVE.is_file():
    restore_report = restore_data_archive(DATA_ARCHIVE, PROJECT_ROOT)
    print(
        f"restored {DATA_ARCHIVE.name}: samples={restore_report['sample_count']} "
        f"stages={restore_report['sample_counts']}"
    )
else:
    print(f"未找到 {DATA_ARCHIVE.name}，使用已有 dataset 目录")


## 2. 数据与消融配置


In [ ]:
from datetime import datetime

from ramanv2.core.config import build_config
from ramanv2.data.datasets import load_dataset_registry
from ramanv2.data.profiles import resolve_selection

# 数据集与层级：PB 的 level_1 是属级全局 head（22 类）。
DATASET_ID = "PB"
LEVEL_NAME = "level_1"
GLOBAL_HEAD_ENABLE = True
# 训练 seed；切分 seed 单独给，保证四组共享同一份 train/val。
SEED = 42
GROUP_SEED = 42
# direct 主干 token 数是 CNN 路径的 4 倍；显存不足时调小这里（四组会一起改）。
BATCH_SIZE = None
# 填写整数覆盖训练轮数；None 使用 config 默认值。
EPOCHS = None
# 每次运行生成新目录：output/<数据集目录>/<RUN_TAG>_<变体>_<seed>。
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
# 只改这两个字段；其余结构、损失与优化参数四组完全一致。
VARIANTS = (
    ("cnn", {"backbone_type": "cnn", "encoder_type": "none"}),
    ("direct_transformer", {"backbone_type": "direct", "encoder_type": "transformer"}),
    ("direct_lstm", {"backbone_type": "direct", "encoder_type": "lstm"}),
    ("cnn_lstm", {"backbone_type": "cnn", "encoder_type": "lstm"}),
)

dataset_root = PROJECT_ROOT / "dataset"
registry = load_dataset_registry(dataset_root)
for item in registry.list_datasets():
    print(f"{item.dataset_id:6} {item.display_name}  dir={item.directory.name}  levels={item.hierarchy.level_names}")

selection = resolve_selection(DATASET_ID, None, PROJECT_ROOT)
spec = selection.spec
profile = selection.profile
dataset_dir = selection.dataset_dir

base_values = {
    "dataset_id": spec.dataset_id,
    "profile_id": profile.profile_id,
    "label_schema": selection.hierarchy.label_schema,
    "bad_bands": spec.bad_bands,
    "seed": int(SEED),
}
if BATCH_SIZE is not None:
    base_values["batch_size"] = int(BATCH_SIZE)
if EPOCHS is not None:
    base_values["epochs"] = int(EPOCHS)
config = build_config(base_values)
output_root = PROJECT_ROOT / "output" / spec.directory.name

print(f"\nselected = {spec.dataset_id} / {spec.display_name}  hierarchy={selection.hierarchy.level_names}")
print(f"dataset_dir = {dataset_dir}")
print(f"batch_size={config.training.batch_size}  epochs={config.training.epochs}  amp={config.execution.use_amp}")
for name, values in VARIANTS:
    print(f"  {name:20} backbone={values['backbone_type']:6} encoder={values['encoder_type']}")


## 3. 构建训练集


In [ ]:
from ramanv2.data.builders.train_test import build_train
from ramanv2.data.count import count_dataset

# 首次运行或数据更新后打开；四组消融共用同一份 train。
BUILD_TRAIN_ENABLE = False
SHOW_COUNTS_ENABLE = True

train_dir = dataset_dir / profile.root_train_clean
if BUILD_TRAIN_ENABLE:
    build_train(profile, dataset_dir, input_config=config.input, hierarchy=selection.hierarchy)
if SHOW_COUNTS_ENABLE:
    if train_dir.is_dir():
        _tree, sample_count = count_dataset(train_dir)
        print(f"[train] samples={sample_count}  dir={train_dir}")
    else:
        print(f"训练目录不存在：{train_dir}")


## 4. 训练与验证集评估


In [ ]:
import json

from ramanv2.core.run_context import open_experiment_context
from ramanv2.data.runtime.index import DatasetIndex
from ramanv2.evaluation.model_eval import evaluate_model_run
from ramanv2.training.scope import resolve_training_parent_ids
from ramanv2.training.session import run_training_scope
from ramanv2.training.split import save_split_files, split_by_lowest_level_ratio

TRAIN_ENABLE = True
CPU_ENABLE = False

if TRAIN_ENABLE:
    if not train_dir.is_dir():
        raise FileNotFoundError(f"训练目录不存在：{train_dir}")
    dataset_index = DatasetIndex(train_dir, selection.hierarchy, load_intensity_enable=False)
    train_indices, val_indices = split_by_lowest_level_ratio(
        dataset_index,
        train_ratio=float(config.training.train_split),
        seed=int(GROUP_SEED),
    )
    # 与实验组一致按索引排序后再落盘，保证切分文件可逐字节比对。
    train_indices = sorted(int(index) for index in train_indices)
    val_indices = sorted(int(index) for index in val_indices)
    parent_ids = resolve_training_parent_ids(
        dataset_index,
        LEVEL_NAME,
        None,
        GLOBAL_HEAD_ENABLE,
    )
    print(f"train={len(train_indices)}  val={len(val_indices)}  parent_ids={parent_ids}")
    results = {}
    for name, variant_values in VARIANTS:
        values = dict(base_values)
        values.update(variant_values)
        variant_config = build_config(values)
        experiment_dir = output_root / f"{RUN_TAG}_{name}_{int(SEED)}"
        # 四组写入同一份 train/val 清单，只让模型结构不同。
        open_experiment_context(experiment_dir, variant_config)
        save_split_files(dataset_index, train_indices, val_indices, experiment_dir)
        print(
            f"\n===== {name}: backbone={variant_values['backbone_type']} "
            f"encoder={variant_values['encoder_type']} -> {experiment_dir} ====="
        )
        scope = run_training_scope(
            variant_config,
            experiment_dir,
            LEVEL_NAME,
            parent_ids,
            GLOBAL_HEAD_ENABLE,
        )
        slot_dir = scope.last_slot_dir
        result_dir = evaluate_model_run(slot_dir, LEVEL_NAME, "cpu" if CPU_ENABLE else None)
        summary = json.loads((result_dir / "metrics.json").read_text(encoding="utf-8"))["summary"]
        results[name] = {
            "experiment_dir": experiment_dir,
            "slot_dir": slot_dir,
            "result_dir": result_dir,
            "summary": summary,
        }
        print(
            f"val accuracy={summary['accuracy'] * 100:.2f}%  "
            f"macro_f1={summary['macro_f1'] * 100:.2f}%  support={summary['support']}"
        )


## 5. 消融对比汇总


In [ ]:
import torch

from ramanv2.common.file_ops import file_sha256

if not results:
    raise ValueError("请先运行训练与评估格")

split_hashes = {
    name: file_sha256(item["experiment_dir"] / "train_split.json")
    for name, item in results.items()
}
print("train_split sha256:", " | ".join(f"{name}={value[:12]}" for name, value in split_hashes.items()))
print(f"切分一致：{len(set(split_hashes.values())) == 1}")

buffer_suffixes = ("running_mean", "running_var", "num_batches_tracked")
header = f"{'variant':<20}{'accuracy':>10}{'macro_f1':>10}{'macro_recall':>13}{'trainable':>11}"
print(header)
print("-" * len(header))
for name, item in results.items():
    summary = item["summary"]
    state = torch.load(item["slot_dir"] / "model.pt", map_location="cpu")
    trainable = sum(
        tensor.numel()
        for key, tensor in state.items()
        if key != "pos_encoder.pe" and not key.endswith(buffer_suffixes)
    )
    print(
        f"{name:<20}{summary['accuracy'] * 100:>9.2f}%"
        f"{summary['macro_f1'] * 100:>9.2f}%"
        f"{summary['macro_recall'] * 100:>12.2f}%"
        f"{trainable:>11,}"
    )


## 6. 独立测试集对比


In [ ]:
import json
from pathlib import Path

from ramanv2.evaluation.independent_test import evaluate_test_set

TEST_EVAL_ENABLE = True
# 每个测试文件夹抽样多少条；None 表示全量测试。
TEST_SAMPLE_LIMIT = None

test_dir = dataset_dir / profile.root_test
if TEST_EVAL_ENABLE:
    if not test_dir.is_dir():
        raise FileNotFoundError(f"未找到独立测试集：{test_dir}")
    if not results:
        raise ValueError("请先运行训练与评估格")

    test_results = {}
    for name, item in results.items():
        report = evaluate_test_set(
            item["slot_dir"],
            "cpu" if CPU_ENABLE else None,
            test_dir,
            TEST_SAMPLE_LIMIT,
        )
        result_dir = Path(report["result_dir"])
        summary = json.loads((result_dir / "metrics.json").read_text(encoding="utf-8"))["summary"]
        test_results[name] = {"result_dir": result_dir, "summary": summary}
        print(
            f"{name:<20} accuracy={summary['accuracy'] * 100:.2f}%  "
            f"macro_f1={summary['macro_f1'] * 100:.2f}%  support={summary['support']}"
        )

    header = f"{'variant':<20}{'accuracy':>10}{'macro_f1':>10}{'macro_recall':>13}{'support':>9}"
    print("\n" + header)
    print("-" * len(header))
    for name, item in test_results.items():
        summary = item["summary"]
        print(
            f"{name:<20}{summary['accuracy'] * 100:>9.2f}%"
            f"{summary['macro_f1'] * 100:>9.2f}%"
            f"{summary['macro_recall'] * 100:>12.2f}%"
            f"{summary['support']:>9}"
        )


## 7. 打包实验结果


In [ ]:
from ramanv2.packaging.experiment_archive import publish_experiment_archive

# False 时只打印实验目录，不生成压缩包。
PACKAGE_RESULT_ENABLE = True

experiment_dirs = sorted(output_root.glob(f"{RUN_TAG}_*_{int(SEED)}"))
if not experiment_dirs:
    raise FileNotFoundError(f"未找到本次消融实验目录：{output_root}/{RUN_TAG}_*_{int(SEED)}")
for experiment_dir in experiment_dirs:
    print(f"=== {experiment_dir.name} ===")
    publish_experiment_archive(experiment_dir, package_enable=PACKAGE_RESULT_ENABLE)
